“A customer says GitLab is down. What do you check first?”

ill check if this for one project or all projects, if this is something started recently or happening from a while..

then ill check, the file, /etc/gitlab/gitlab.rb, this is single source of truth.

sudo cat /etc/gitlab/gitlab.rb | grep "external_url" and similarly for ports, resource limits, then sudo gitlab-ctl reconfigure
sudo gitlab-ctl restart

then I will check df -h , free -g for any resource issues if the above is all good.

Good instincts on **scope** and **recent change**, but for “GitLab is down,” you should **not start with `gitlab.rb` or reconfigure/restart first**.

That jumps to config changes before confirming the symptom.

A stronger answer:

> “First I want to confirm what ‘down’ means: is the UI unreachable, are Git pushes failing too, and is it all users or only one project. Then I’d check host health and GitLab service health before changing any config.”

Then:

```bash
sudo gitlab-ctl status
curl -I http://localhost
ss -tulpn | grep -E ':80|:443'
df -h
free -m
sudo gitlab-ctl tail
```

Why this is better:

* `gitlab.rb` is config, not the first place to look for a live outage
* `gitlab-ctl reconfigure` and `restart` are changes, so do them only after evidence
* for “down,” first prove whether it is:

  * web service issue
  * overall GitLab service issue
  * host resource issue
  * bad recent config change

Better spoken version:

> “I’d first clarify whether this is instance-wide or limited to one function. Then I’d check if the host is healthy, whether core GitLab services are running, whether ports 80/443 are listening, and what the logs show. If I find evidence of a recent config issue, then I’d inspect `gitlab.rb` and only make the smallest safe change needed.”

I’d rate your answer **6.5/10**.
Good start on scope, but the main fix is: **check health and logs before config edits or restart**.

A very strong one-line order to remember:

**scope → host health → service health → logs → config only if evidence points there**
